In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import torch

from src.model import register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset
from src.train.train import processor_init, model_init, ModelArguments, LoraArguments, WhisperAccentTrainingArguments
from src.train.trainer import WhisperAccentTrainer
register_whisper_accent()

In [ ]:
import datetime

run_name = (
    f"whisper-accent-tiny-en-lora-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
)
output_dir = "/workspace/whisper-accent-tiny.en"
model_args = ModelArguments(
    model_type="whisper",
    base_model_name_or_path="openai/whisper-tiny.en",
    is_multilingual=False,
)
lora_args = LoraArguments()
training_args = WhisperAccentTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_steps=500,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=100,
    bf16=True,
    fp16=False,
    eval_steps=100,
    run_name=run_name,
    optim="adamw_torch",
    report_to=["tensorboard"],
    push_to_hub=True,
    logging_first_step=True,
    hub_model_id="mavleo96/whisper-accent-tiny.en",
    hub_strategy="all_checkpoints",
    gradient_checkpointing=False,
    predict_with_generate=True,
    remove_unused_columns=False,
)

In [ ]:
processor = processor_init(model_args)
model = model_init(model_args, lora_args, processor)
model.print_trainable_parameters()

In [ ]:
collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
)

train_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="train",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    shuffle=True,
    num_proc=16,
)

eval_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="validation",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    shuffle=False,
    num_proc=16,
)


In [ ]:
trainer = WhisperAccentTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
)


In [ ]:
trainer.train()


In [ ]:
# trainer.model.merge_and_unload().save_pretrained(f"{output_dir}")
# processor.save_pretrained(f"{output_dir}")
# trainer.push_to_hub()

In [ ]:
# accent_embeddings = model.model.decoder.embed_tokens.weight[
#     list(model.generation_config.accent_to_id.values()), :
# ]
# accent_embeddings.shape
# torch.nn.functional.cosine_similarity(accent_embeddings, accent_embeddings, dim=1)